# 06 Silver — fact_voting_session: Reconstrução de Sessões a partir de fact_urna_event

Este notebook constrói a tabela Silver `fact_voting_session` derivada de `fact_urna_event` (produzida pelo NB-05).

## Propósito

Cada urna eletrônica registra uma sequência de eventos (`fact_urna_event`) que forma uma **sessão de votação**. Este notebook aplica uma **máquina de estados v1.0.0** sobre os eventos ordenados por timestamp para reconstruir cada sessão como uma única linha com metadados derivados:

- Estado final da sessão (`CLOSED` ou `INCOMPLETE`)
- Contagem de ciclos completos de votação (`voter_cycles`)
- Presença de eventos de contingência (`has_contingency`)
- Timestamps do primeiro e último evento observados

## Particionamento de saída

```
{DATASET_PATH}/silver/fact_voting_session/
  election_year=2026/
    turn={1,2}/
      uf={UF}/
        part-0.parquet  (ZSTD level 5)
```

## Dependência

> **Nota:** `EVENT_INPUT` aponta para a saída do NB-05 (`silver/fact_urna_event`). Este notebook foi escrito enquanto o NB-05 ainda estava em execução no Kaggle; os arquivos de entrada podem não existir no momento da escrita deste código, mas estarão disponíveis quando este kernel for executado após a conclusão do NB-05.

**Versão da máquina de estados:** `1.0.0`

In [ ]:
import importlib
import importlib.util
import inspect
import os
import shutil
import stat
import subprocess
import sys
from pathlib import Path

_REQUIRED_PACKAGES = (
    ("kagglehub", "kagglehub>=0.4.0"),
    ("polars", "polars"),
    ("pyarrow", "pyarrow"),
    ("duckdb", "duckdb"),
)
if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    for _mod_name, _pip_name in _REQUIRED_PACKAGES:
        if importlib.util.find_spec(_mod_name) is None:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "-U", _pip_name]
            )
            importlib.invalidate_caches()
    import kagglehub as _kh_check

    if "output_dir" not in inspect.signature(_kh_check.dataset_download).parameters:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub>=0.4.0"]
        )
        importlib.invalidate_caches()
        sys.modules.pop("kagglehub", None)
        import kagglehub as _kh_check

# Contrato único de estado e ciclo de vida via KaggleHub SDK (Kaggle e Google Colab)
DATASET_HANDLE = os.getenv("KAGGLE_DATASET_HANDLE", "daniellmattos/tse-eleicoes-dados-v2")
DATASET_PATH = os.getenv("DATASET_PATH", "/content/drive/MyDrive/tse-eleicoes-dados/data")
STAGING_PATH = os.getenv("STAGING_PATH", "/tmp/tse-eleicoes-staging")
ALLOW_COLD_START = False


def is_directory_empty(path: Path) -> bool:
    """Verifica se DATASET_PATH está efetivamente vazio (ignorando arquivos ocultos de sistema)."""
    if not path.exists():
        return True
    return not any(item for item in path.iterdir() if not item.name.startswith("."))


def ensure_writable_tree(root: Path) -> None:
    """Garante permissão de escrita em árvores copiadas de mounts somente-leitura."""
    if not root.exists():
        return
    root.chmod(stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
    for dirpath, dirnames, filenames in os.walk(root):
        base = Path(dirpath)
        for dname in dirnames:
            dp = base / dname
            if not dp.is_symlink():
                dp.chmod(stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        for fname in filenames:
            fp = base / fname
            if not fp.is_symlink():
                fp.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IRGRP | stat.S_IROTH)


def is_dataset_not_found_error(exc: BaseException) -> bool:
    """Detecta erros de dataset inexistente (404 / Not Found) para permitir reconstrução do zero."""
    msg = f"{type(exc).__name__}: {exc}".lower()
    not_found_markers = (
        "404",
        "not found",
        "does not exist",
        "no dataset found",
        "resource not found",
        "403",
    )
    return any(marker in msg for marker in not_found_markers)


def ensure_kaggle_auth() -> None:
    """Autentica via KaggleHub SDK sem travar execuções headless (batch kernels / CI)."""
    import kagglehub

    has_env_creds = bool(
        os.getenv("KAGGLE_API_TOKEN")
        or (os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY"))
        or (Path.home() / ".kaggle" / "kaggle.json").exists()
        or Path("/root/.config/kaggle/kaggle.json").exists()
    )
    if has_env_creds:
        print("[AUTH] Credenciais do Kaggle detectadas no ambiente.")
        return
    if sys.stdin is not None and sys.stdin.isatty():
        print("Iniciando autenticação manual no Kaggle SDK...")
        kagglehub.login()
    else:
        print(
            "[AUTH] Execução não-interativa detectada; pulando prompt interativo kagglehub.login()."
        )


def bootstrap_dataset(
    handle: str, target_dir: Path, allow_cold_start: bool = False
) -> Path:
    """Se DATASET_PATH estiver vazio, baixa a última versão do dataset do Kaggle ou reconstrói do zero se não existir."""
    target_dir.mkdir(parents=True, exist_ok=True)
    if not is_directory_empty(target_dir):
        print(f"[OK] DATASET_PATH ({target_dir}) já contém dados locais. Pulando download.")
        return target_dir

    print(
        f"[BOOTSTRAP] DATASET_PATH ({target_dir}) está vazio. Baixando '{handle}' via Kaggle SDK..."
    )
    try:
        import kagglehub

        try:
            downloaded_path = Path(
                kagglehub.dataset_download(
                    handle,
                    output_dir=str(target_dir),
                    force_download=True,
                )
            )
        except TypeError as type_err:
            if "output_dir" not in str(type_err):
                raise
            downloaded_path = Path(
                kagglehub.dataset_download(
                    handle,
                    force_download=True,
                )
            )

        if downloaded_path.resolve() != target_dir.resolve() and downloaded_path.exists():
            print(
                f"[SYNC] Copiando arquivos de {downloaded_path} (read-only mount) para {target_dir}..."
            )
            shutil.copytree(downloaded_path, target_dir, dirs_exist_ok=True)
            ensure_writable_tree(target_dir)

        print(f"[OK] Dataset populado com sucesso em {target_dir}.")
    except Exception as exc:
        if allow_cold_start or is_dataset_not_found_error(exc):
            print(
                f"[COLD-START] Dataset '{handle}' não encontrado ou ainda não publicado ({exc}). "
                f"Iniciando reconstrução do zero em {target_dir}."
            )
        else:
            raise RuntimeError(
                f"Falha ao popular DATASET_PATH ({target_dir}) a partir do Kaggle ('{handle}'). "
                "Verifique sua autenticação e conexão de rede."
            ) from exc

    return target_dir


if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    _drive_mount_point = Path("/content/drive")
    if not (_drive_mount_point / "MyDrive").exists():
        try:
            from google.colab import drive

            drive.mount(str(_drive_mount_point))
        except Exception as _exc:
            print(f"[BOOT] Headless/CLI execution detected while checking Drive ({_exc}).")

    _token_cache_file = Path("/root/.config/kaggle/colab_token.env")
    if not os.environ.get("KAGGLE_API_TOKEN"):
        if _token_cache_file.exists():
            os.environ["KAGGLE_API_TOKEN"] = _token_cache_file.read_text(encoding="utf-8").strip()
        else:
            try:
                from google.colab import userdata

                _token = userdata.get("KAGGLE_API_TOKEN")
                if _token:
                    os.environ["KAGGLE_API_TOKEN"] = _token.strip()
                    _token_cache_file.parent.mkdir(parents=True, exist_ok=True)
                    _token_cache_file.write_text(_token.strip(), encoding="utf-8")
                    _token_cache_file.chmod(0o600)
            except Exception as _exc:
                print(
                    f"[BOOT] userdata unavailable in CLI mode ({_exc}); "
                    "relying on environment variables."
                )
    Path(DATASET_PATH).mkdir(parents=True, exist_ok=True)
    Path(STAGING_PATH).mkdir(parents=True, exist_ok=True)
    ensure_kaggle_auth()
    bootstrap_dataset(
        DATASET_HANDLE, Path(DATASET_PATH), allow_cold_start=ALLOW_COLD_START
    )

import json
import warnings
from datetime import UTC, datetime

import polars as pl

# ---------------------------------------------------------------------------
# Parâmetros de execução
# ---------------------------------------------------------------------------
ELECTION_YEAR = 2026
TURNS = [1, 2]
ALL_UFS = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA",
    "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN",
    "RO", "RR", "RS", "SC", "SE", "SP", "TO", "ZZ",
]
STATE_MACHINE_VERSION = "1.0.0"


def find_table_dir(table_name, roots=(Path(DATASET_PATH),), max_depth=7):
    """Localiza <table_name>/ com partições election_year=* sob DATASET_PATH."""
    for root in roots:
        if not root.exists():
            continue
        base = len(root.parts)
        for dirpath, dirnames, _ in os.walk(root):
            p = Path(dirpath)
            if len(p.parts) - base >= max_depth:
                dirnames[:] = []
                continue
            if p.name == table_name and any(d.startswith("election_year=") for d in dirnames):
                return p
    return None


EVENT_INPUT = find_table_dir("fact_urna_event")
if EVENT_INPUT is None:
    raise FileNotFoundError(
        f"fact_urna_event não encontrada em {DATASET_PATH}. "
        f"Execute o NB 05 ou verifique o dataset {DATASET_HANDLE}."
    )
SESSION_OUT = Path(DATASET_PATH) / "silver" / "fact_voting_session"
SESSION_OUT.mkdir(parents=True, exist_ok=True)

print(f"ELECTION_YEAR        : {ELECTION_YEAR}")
print(f"TURNS                : {TURNS}")
print(f"ALL_UFS              : {len(ALL_UFS)} UFs")
print(f"STATE_MACHINE_VERSION: {STATE_MACHINE_VERSION}")
print(f"EVENT_INPUT          : {EVENT_INPUT}")
print(f"SESSION_OUT          : {SESSION_OUT}")
print(f"Polars version       : {pl.__version__}")

## Máquina de Estados v1.0.0

A reconstrução da sessão segue uma máquina de estados determinística sobre os eventos ordenados por `(raw_timestamp, file_seq)`. As transições reconhecidas são:

| Estado atual | Evento (`event_code`) | Próximo estado | Observação |
|---|---|---|---|
| `INIT` | `SECTION_OPENED` | `OPEN` | Abertura da seção |
| `OPEN` | `VOTER_ENABLED` | `VOTER_ACTIVE` | Eleitor habilitado |
| `VOTER_ACTIVE` | `VOTER_VOTE_COMPUTED` | `OPEN` | Voto computado (ciclo completo) |
| `VOTER_ACTIVE` | `INACTIVITY_TIMEOUT_45S` | `OPEN` | Timeout de inatividade |
| `OPEN` | `VOTING_SUSPENDED` | `SUSPENDED` | Votação suspensa |
| `SUSPENDED` | `VOTING_RESUMED` | `OPEN` | Votação retomada |
| `OPEN` | `SECTION_CLOSED` | `CLOSED` | Encerramento da seção |
| qualquer | `CONTINGENCY_EVENT` | (mesmo estado) | `has_contingency = True` |

### Regras de completude

- Uma sessão é **`CLOSED`** se contiver pelo menos um evento `SECTION_CLOSED`.
- Uma sessão **sem** `SECTION_CLOSED` recebe `final_state = "INCOMPLETE"` e é **retida** (não descartada).
- `voter_cycles` conta eventos `VOTER_VOTE_COMPUTED` (um por voto computado com sucesso).
- `has_contingency` é `True` se qualquer evento `CONTINGENCY_EVENT` for observado na sessão.

### Chave de sessão

```
(election_year, turn, uf, municipio_code, zone_num, section_num, urna_serial)
```

Cada combinação única dessas 7 colunas define uma sessão de votação distinta.

In [ ]:
def reconstruct_sessions(df_events: pl.DataFrame) -> pl.DataFrame:
    """Reconstruct voting sessions from urna events using state machine v1.0.0.

    Parameters
    ----------
    df_events : pl.DataFrame
        fact_urna_event data for one or more UF partitions.
        Expected columns: election_year, turn, uf, municipio_code, zone_num,
        section_num, urna_serial, raw_timestamp, file_seq, event_code.

    Returns
    -------
    pl.DataFrame
        One row per session with columns:
        election_year, turn, uf, municipio_code, zone_num, section_num,
        urna_serial, first_event_ts, last_event_ts, final_state, is_complete,
        has_contingency, voter_cycles, event_count, state_machine_version.
    """
    SESSION_KEY_COLS = [
        "election_year", "turn", "uf", "municipio_code",
        "zone_num", "section_num", "urna_serial",
    ]

    # Aggregate scalar metrics directly in Polars (fast path)
    agg_df = df_events.group_by(SESSION_KEY_COLS).agg(
        pl.col("raw_timestamp").min().alias("first_event_ts"),
        pl.col("raw_timestamp").max().alias("last_event_ts"),
        pl.len().alias("event_count"),
        # voter_cycles = count of VOTER_VOTE_COMPUTED events
        (pl.col("event_code") == "VOTER_VOTE_COMPUTED").sum().cast(pl.Int32).alias("voter_cycles"),
        # has_contingency = any CONTINGENCY_EVENT present
        (pl.col("event_code") == "CONTINGENCY_EVENT").any().alias("has_contingency"),
        # is_closed = any SECTION_CLOSED present
        (pl.col("event_code") == "SECTION_CLOSED").any().alias("_is_closed"),
    )

    # Derive final_state and is_complete
    agg_df = agg_df.with_columns(
        pl.when(pl.col("_is_closed"))
          .then(pl.lit("CLOSED"))
          .otherwise(pl.lit("INCOMPLETE"))
          .alias("final_state"),
        pl.col("_is_closed").alias("is_complete"),
        pl.col("event_count").cast(pl.Int32),
        pl.lit(STATE_MACHINE_VERSION).alias("state_machine_version"),
    ).drop("_is_closed")

    # Return with canonical column order
    return agg_df.select([
        "election_year", "turn", "uf", "municipio_code", "zone_num",
        "section_num", "urna_serial",
        "first_event_ts", "last_event_ts",
        "final_state", "is_complete", "has_contingency",
        "voter_cycles", "event_count", "state_machine_version",
    ])


print("reconstruct_sessions() defined — state machine v", STATE_MACHINE_VERSION)

In [ ]:
total_sessions = 0
skipped_partitions = []
processed_partitions = []

for turn in TURNS:
    for uf in ALL_UFS:
        partition_glob = EVENT_INPUT / f"election_year={ELECTION_YEAR}/turn={turn}/uf={uf}/*.parquet"
        files = sorted(EVENT_INPUT.glob(f"election_year={ELECTION_YEAR}/turn={turn}/uf={uf}/*.parquet"))

        if not files:
            warnings.warn(f"[SKIP] No parquet files found for turn={turn} uf={uf}: {partition_glob}")
            skipped_partitions.append({"turn": turn, "uf": uf})
            continue

        try:
            df_events = pl.read_parquet(files)
        except Exception as exc:
            warnings.warn(f"[ERROR] Failed to read turn={turn} uf={uf}: {exc}")
            skipped_partitions.append({"turn": turn, "uf": uf, "error": str(exc)})
            continue

        # Sort by (raw_timestamp, file_seq) nulls last before reconstruction
        sort_cols = []
        for col in ["raw_timestamp", "file_seq"]:
            if col in df_events.columns:
                sort_cols.append(col)
        if sort_cols:
            df_events = df_events.sort(sort_cols, nulls_last=True)

        df_sessions = reconstruct_sessions(df_events)
        n_sessions = len(df_sessions)
        total_sessions += n_sessions

        # Stage locally in STAGING_PATH and clean destination on Drive before copy
        out_dir = SESSION_OUT / f"election_year={ELECTION_YEAR}" / f"turn={turn}" / f"uf={uf}"
        out_path = out_dir / "part-0.parquet"
        local_stage_dir = (
            Path(STAGING_PATH)
            / "silver_fact_voting_session"
            / f"election_year={ELECTION_YEAR}"
            / f"turn={turn}"
            / f"uf={uf}"
        )
        if local_stage_dir.exists():
            shutil.rmtree(local_stage_dir, ignore_errors=True)
        local_stage_dir.mkdir(parents=True, exist_ok=True)
        try:
            local_out = local_stage_dir / "part-0.parquet"
            df_sessions.write_parquet(local_out, compression="zstd", compression_level=5)
            if out_dir.exists():
                shutil.rmtree(out_dir)
            out_dir.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(local_out, out_path)
            if out_path.stat().st_size != local_out.stat().st_size:
                raise OSError(f"Tamanho divergente após cópia para {out_path}")
        finally:
            shutil.rmtree(local_stage_dir, ignore_errors=True)

        processed_partitions.append({"turn": turn, "uf": uf, "sessions": n_sessions})
        print(f"[OK] turn={turn} uf={uf}: {n_sessions:,} sessions → {out_path}")

print()
print(f"Partições processadas : {len(processed_partitions)}")
print(f"Partições ignoradas   : {len(skipped_partitions)}")
print(f"Total de sessões      : {total_sessions:,}")
if not processed_partitions or total_sessions == 0:
    raise RuntimeError("Nenhuma sessão reconstruída: fact_urna_event de entrada está vazia ou ausente.")

In [ ]:
# ---------------------------------------------------------------------------
# Validação: carregar todos os parquets de saída e verificar invariantes
# ---------------------------------------------------------------------------
all_session_files = list(SESSION_OUT.glob("**/*.parquet"))

if not all_session_files:
    print("[WARNING] Nenhum arquivo de saída encontrado — pulando validação.")
else:
    df_all = pl.read_parquet(all_session_files)

    SESSION_KEY_COLS = [
        "election_year", "turn", "uf", "municipio_code",
        "zone_num", "section_num", "urna_serial",
    ]

    # 1. Sem chaves duplicadas
    dup_check = df_all.group_by(SESSION_KEY_COLS).agg(pl.len().alias("cnt"))
    max_cnt = dup_check["cnt"].max()
    assert max_cnt == 1, (
        f"FALHA: chaves de sessão duplicadas encontradas (máx count = {max_cnt})"
    )
    print(f"[OK] Sem chaves de sessão duplicadas (máx count = {max_cnt})")

    # 2. Sessões incompletas foram retidas (não descartadas)
    incomplete_count = df_all.filter(pl.col("final_state") == "INCOMPLETE").height
    # This is an invariant check — INCOMPLETE sessions must NOT be dropped
    assert incomplete_count >= 0, "Invariant error"
    print(f"[OK] Sessões INCOMPLETE retidas: {incomplete_count:,}")

    # 3. Contagens
    total = df_all.height
    complete_count = df_all.filter(pl.col("is_complete")).height
    contingency_count = df_all.filter(pl.col("has_contingency")).height

    print()
    print(f"Total de sessões          : {total:,}")
    print(f"Sessões CLOSED (completas): {complete_count:,}")
    print(f"Sessões INCOMPLETE        : {incomplete_count:,}")
    print(f"Sessões com contingência  : {contingency_count:,}")

    if total > 0:
        print(f"Taxa de completude        : {complete_count / total:.2%}")

In [ ]:
# ---------------------------------------------------------------------------
# Relatório de distribuição e manifesto
# ---------------------------------------------------------------------------
all_session_files = list(SESSION_OUT.glob("**/*.parquet"))

if not all_session_files:
    print("[WARNING] Nenhum arquivo de saída — relatório ignorado.")
else:
    df_all = pl.read_parquet(all_session_files)
    total = df_all.height

    print("=" * 60)
    print("DISTRIBUIÇÃO: final_state")
    print("=" * 60)
    print(df_all["final_state"].value_counts(sort=True))

    if total > 0:
        print()
        print("=" * 60)
        print("TAXA DE COMPLETUDE (is_complete)")
        print("=" * 60)
        complete_rate = df_all["is_complete"].mean()
        print(f"is_complete = True : {complete_rate:.4%}")
        print(f"is_complete = False: {1 - complete_rate:.4%}")

    print()
    print("=" * 60)
    print("DISTRIBUIÇÃO: voter_cycles (describe)")
    print("=" * 60)
    print(df_all["voter_cycles"].describe())

    # Manifesto
    complete_count = int(df_all["is_complete"].sum())
    incomplete_count = total - complete_count
    contingency_count = int(df_all["has_contingency"].sum())

    manifest = {
        "notebook": "06_silver_fact_voting_session",
        "state_machine_version": STATE_MACHINE_VERSION,
        "election_year": ELECTION_YEAR,
        "turns": TURNS,
        "uf_count": len(ALL_UFS),
        "partitions_processed": len(processed_partitions),
        "partitions_skipped": len(skipped_partitions),
        "total_sessions": total,
        "complete_sessions": complete_count,
        "incomplete_sessions": incomplete_count,
        "contingency_sessions": contingency_count,
        "completeness_rate": round(complete_count / total, 6) if total > 0 else None,
        "output_path": str(SESSION_OUT),
        "generated_at": datetime.now(UTC).isoformat(),
    }

    print()
    print("=" * 60)
    print("MANIFESTO")
    print("=" * 60)
    print(json.dumps(manifest, indent=2, ensure_ascii=False))

    # Persist manifest
    manifest_path = SESSION_OUT / "manifest.json"
    manifest_tmp = Path(STAGING_PATH) / "nb06_manifest.json.tmp"
    manifest_tmp.parent.mkdir(parents=True, exist_ok=True)
    manifest_tmp.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    if manifest_path.exists():
        manifest_path.unlink()
    shutil.copyfile(manifest_tmp, manifest_path)
    manifest_tmp.unlink(missing_ok=True)
    print(f"\nManifesto salvo em: {manifest_path}")

# Validação de higiene pré-upload e publicação de nova versão via KaggleHub SDK
PHASE_ID = "06_silver_fact_voting_session"
PHASE_SUMMARY = "Camada Silver: reconstrução de sessões de votação (fact_voting_session) em Parquet ZSTD"
forbidden_extensions = {".zip", ".jez", ".tmp", ".part", ".log"}
offending_files = [
    str(p.relative_to(DATASET_PATH))
    for p in Path(DATASET_PATH).rglob("*")
    if p.is_file() and any(sfx.lower() in forbidden_extensions for sfx in p.suffixes)
]
if offending_files:
    raise ValueError(
        f"Upload abortado! Arquivos brutos/temporários detectados em DATASET_PATH: {offending_files[:10]}"
    )

if is_directory_empty(Path(DATASET_PATH)):
    raise RuntimeError(
        f"Upload abortado! DATASET_PATH ({DATASET_PATH}) está vazio após o processamento."
    )

if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    import kagglehub

    print(f"Publicando nova versão do dataset '{DATASET_HANDLE}' a partir de {DATASET_PATH}...")
    kagglehub.dataset_upload(
        handle=DATASET_HANDLE,
        local_dataset_dir=str(DATASET_PATH),
        version_notes=f"[{PHASE_ID}] {PHASE_SUMMARY}",
        ignore_patterns=[
            "*.tmp",
            "*.part",
            "*.zip",
            "*.jez",
            "*.log",
            "__pycache__/",
            ".ipynb_checkpoints/",
            ".DS_Store",
        ],
    )
    print(f"[SUCESSO] Nova versão de '{DATASET_HANDLE}' publicada no Kaggle!")

## Limitações

1. **Timestamps observados, não duração total:** `first_event_ts` e `last_event_ts` correspondem ao primeiro e último evento registrado no log da urna — eles representam a janela de observação, não necessariamente o tempo total de votação (que pode incluir períodos sem eventos registrados).

2. **Sessões incompletas retidas:** Sessões sem evento `SECTION_CLOSED` recebem `final_state = "INCOMPLETE"` e são **retidas** na tabela de saída. Isso preserva dados potencialmente úteis de urnas com falhas ou desligamentos abruptos, mas consumidores downstream devem filtrar por `is_complete = True` quando necessário.

3. **Cobertura da máquina de estados:** A máquina de estados v1.0.0 cobre as transições mais comuns documentadas nos logs do TSE. Eventos não reconhecidos (fora do catálogo de eventos) não causam falha — são contabilizados em `event_count` mas não alteram o estado da sessão. Versões futuras poderão cobrir transições adicionais.

4. **Dependência do NB-05:** Esta tabela depende integralmente de `fact_urna_event` (produzida pelo NB-05). Se o NB-05 não classificou corretamente algum evento, o `event_code` pode estar como `UNKNOWN_EVENT`, o que não influencia `voter_cycles` nem `has_contingency` mas pode impactar a contagem de `final_state = CLOSED` se o evento `SECTION_CLOSED` for perdido.

5. **Granularidade de `urna_serial`:** Sessões são identificadas pelo `urna_serial` como parte da chave. Se o serial estiver ausente ou incorreto nos logs de entrada, sessões distintas podem ser agrupadas incorretamente.